# Markov Network Workflow

This notebook shows the refactored array-based workflow in `markov_network.py`.
It covers three steps:

1. Prepare feature arrays, optional covariates, and an outcome vector.
2. Fit the Markov network with Graphical Lasso.
3. Run outcome permutation testing and write the results to CSV.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent and not (workspace_root / "markov_network.py").exists():
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from markov_network import (
    MarkovNetworkConfig,
    MarkovNetworkCSVWriter,
    MarkovNetworkEstimator,
    MarkovNetworkPermutationTester,
    PermutationTestConfig,
)


## Example data

The next cell creates synthetic data so the notebook runs without any external files.
Replace it with your own arrays or a DataFrame when you are ready.


In [ ]:
rng = np.random.default_rng(7)
n_subjects = 120

latent_a = rng.normal(size=n_subjects)
latent_b = rng.normal(size=n_subjects)
age = 65 + 8 * rng.normal(size=n_subjects)

features = pd.DataFrame(
    {
        "Atrophy_A": latent_a + 0.20 * rng.normal(size=n_subjects),
        "Atrophy_B": 0.65 * latent_a + 0.30 * rng.normal(size=n_subjects),
        "FC_A": -0.50 * latent_a + 0.40 * latent_b + 0.35 * rng.normal(size=n_subjects),
        "FC_B": 0.75 * latent_b + 0.30 * rng.normal(size=n_subjects),
    }
)
covariates = pd.DataFrame({"Age": age})
outcome = 0.55 * features["Atrophy_A"] - 0.45 * features["FC_B"] + 0.03 * (age - 65)
outcome = outcome + 0.40 * rng.normal(size=n_subjects)

features.head()


## Real-data template

If you already have a CSV, use the pattern below and keep the rest of the notebook unchanged.


In [ ]:
# data = pd.read_csv(workspace_root / "path" / "to" / "your_features.csv")
# outcome_name = "Y"
# covariate_columns = ["Age"]
# feature_columns = [column for column in data.columns if column not in {outcome_name, *covariate_columns}]
# features = data[feature_columns]
# covariates = data[covariate_columns]
# outcome = data[outcome_name]


## Fit the network

You can pass pandas objects or NumPy arrays. The estimator standardizes the design matrix, fits the sparse precision matrix, and returns DataFrames for the key outputs.


In [ ]:
estimator = MarkovNetworkEstimator(
    MarkovNetworkConfig(
        alpha_grid=tuple(np.logspace(-4, 0, 10)),
        edge_threshold=0.0,
        verbose=False,
    )
)

fit_result = estimator.fit(
    features=features,
    outcome=outcome,
    covariates=covariates,
    outcome_name="Y",
)

fit_result.statistics


In [ ]:
fit_result.outcome_edge_table


## Run permutation testing

Permutation testing shuffles the outcome vector, refits the model, and compares observed edge strengths against the null distribution for each edge.


In [ ]:
permutation_tester = MarkovNetworkPermutationTester(
    estimator=estimator,
    config=PermutationTestConfig(
        n_permutations=100,
        alpha=0.05,
        random_state=13,
        report_every=25,
    ),
)

permutation_result = permutation_tester.run(
    features=features,
    outcome=outcome,
    covariates=covariates,
    outcome_name="Y",
)

permutation_result.edge_statistics.head(10)


In [ ]:
permutation_result.significant_outcome_edges


## Write CSV outputs

The writer exports both the full model outputs and the permutation-filtered outputs.


In [ ]:
output_dir = workspace_root / "notebook_outputs" / "markov_network_demo"
writer = MarkovNetworkCSVWriter()
writer.write_fit(fit_result, output_dir)
writer.write_permutation_test(permutation_result, output_dir)

sorted(path.name for path in output_dir.iterdir())
